# Mini Project 01 — Notebook 01: Phân loại ảnh với E-ELAN (Task 1 + 2)

**Môn:** Một số vấn đề chọn lọc về Thị giác máy tính — MAT3563
**Nhóm:**
- Nguyễn Đức Quang — 23001918
- Hà Minh Quang — 23001916

**Nội dung notebook này:**
- Xây dựng **backbone E-ELAN** (nhiều stage, xuất đặc trưng đa tỉ lệ C3/C4/C5 để tái dùng cho phần detection ở Notebook 02).
- **Task 1:** Lưu / nạp weights từ đường dẫn chỉ định, chạy **inference** cho 1 ảnh hoặc cả thư mục, in class + xác suất, xuất CSV.
- **Task 2:** Vẽ đồ thị biến thiên **loss & accuracy**; in **precision + recall** (per-class + macro) và confusion matrix trên tập val có nhãn.

> Chạy tuần tự từ trên xuống (Run All). Notebook này sinh ra `weights/best_cls.pt` và `weights/backbone.pt` làm đầu vào cho Notebook 02.


## 0. Cấu hình môi trường & đường dẫn
Tự nhận biết Colab (mount Google Drive) hay chạy local.

In [ ]:
import os

# --- Tự nhận biết Colab hay local ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/MiniProject01'
    IN_COLAB = True
except Exception:
    # Chạy local trong thư mục project
    ROOT = r'D:/CV nâng cao/Project/Mini Project 01'
    IN_COLAB = False

DATA_CLS = os.path.join(ROOT, 'data', 'classification')  # chứa train/ val/  (cats, dogs, panda)
WEIGHTS  = os.path.join(ROOT, 'weights')
RESULTS  = os.path.join(ROOT, 'results')
os.makedirs(WEIGHTS, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

print('IN_COLAB =', IN_COLAB)
print('ROOT     =', ROOT)
print('DATA_CLS =', DATA_CLS)
print('Tồn tại data?', os.path.isdir(DATA_CLS))

In [ ]:
# Thư viện cần thiết (Colab đã có sẵn phần lớn)
# !pip -q install scikit-learn matplotlib
import math, time, copy, glob
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. CONFIG

In [ ]:
CFG = {
    'data_root': DATA_CLS,
    'train_dir': 'train',
    'val_dir':   'val',          # nếu không có sẽ auto-split từ train
    'auto_split_val_ratio': 0.1,

    'img_size': 224,
    'num_workers': 2,

    'epochs': 20,
    'batch_size': 32,
    'lr': 2e-3,
    'weight_decay': 0.05,
    'label_smoothing': 0.0,
    'amp': True,                 # chỉ dùng khi có CUDA

    # Backbone E-ELAN đa tỉ lệ
    'C_stem': 32,
    'widths': (64, 128, 192, 256),  # stage1(H/4), stage2=C3(H/8), stage3=C4(H/16), stage4=C5(H/32)
    'm': 2.0,
    'g': 2,

    'use_cosine': True,
    'min_lr_scale': 0.05,

    'best_ckpt':     os.path.join(WEIGHTS, 'best_cls.pt'),
    'backbone_ckpt': os.path.join(WEIGHTS, 'backbone.pt'),
    'seed': 42,
    'topk': 3,
}

def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(CFG['seed'])

## 2. Các khối E-ELAN (giữ nguyên tinh thần code mẫu của thầy)
`ConvBNAct`, `channel_shuffle`, `EELANLite` — Expand (group conv) → 4 nhánh depth [0,1,2,3] → channel shuffle → concat → 1×1 fuse + residual.

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, c_in, c_out, k=3, s=1, p=None, g=1, act=True):
        super().__init__()
        if p is None:
            p = (k - 1) // 2
        self.conv = nn.Conv2d(c_in, c_out, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(c_out)
        self.act  = nn.SiLU(inplace=True) if act else nn.Identity()
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

def channel_shuffle(x: torch.Tensor, groups: int) -> torch.Tensor:
    if groups <= 1:
        return x
    b, c, h, w = x.shape
    assert c % groups == 0, 'channels must be divisible by groups'
    x = x.view(b, groups, c // groups, h, w)
    x = x.transpose(1, 2).contiguous()
    x = x.view(b, c, h, w)
    return x

class EELANLite(nn.Module):
    """Minimal E-ELAN block (xem mô tả PDF: expand -> branches -> shuffle -> concat -> fuse + residual)."""
    def __init__(self, C_in, C_out=None, m=2.0, g=2, branch_depths=None, use_skip=True):
        super().__init__()
        assert m >= 1
        self.C_in  = C_in
        self.C_out = C_in if C_out is None else C_out
        self.g = g
        self.use_skip = use_skip and (self.C_out == self.C_in)

        C_exp = int(round(m * C_in))
        C_exp = max(4 * g, (C_exp // (4 * g)) * (4 * g))  # chia hết cho 4*g
        self.C_exp = C_exp

        self.expand = ConvBNAct(C_in, C_exp, k=1, s=1, p=0, g=g, act=True)
        self.num_branches = 4
        self.split_ch = C_exp // self.num_branches
        assert C_exp % self.num_branches == 0

        if branch_depths is None:
            branch_depths = [0, 1, 2, 3]
        self.branch_depths = branch_depths[:self.num_branches]

        branches = []
        for d in self.branch_depths:
            layers = []
            for _ in range(d):
                layers.append(ConvBNAct(self.split_ch, self.split_ch, k=3, s=1, p=1, g=g, act=True))
            branches.append(nn.Sequential(*layers) if layers else nn.Identity())
        self.branches = nn.ModuleList(branches)

        self.fuse = ConvBNAct(C_exp, self.C_out, k=1, s=1, p=0, g=1, act=True)

    def forward(self, x):
        identity = x
        y = self.expand(x)
        chunks = torch.chunk(y, self.num_branches, dim=1)
        outs = [br(ch) for br, ch in zip(self.branches, chunks)]
        outs = [channel_shuffle(t, self.g) for t in outs]
        y = torch.cat(outs, dim=1)
        y = self.fuse(y)
        if self.use_skip and y.shape == identity.shape:
            y = y + identity
        return y

## 3. Backbone E-ELAN đa tỉ lệ + Classifier
So với code gốc (chỉ xuống H/4), backbone này có thêm stage để xuất **C3 (H/8), C4 (H/16), C5 (H/32)** — cần thiết để Notebook 02 gắn Neck (FPN) + Head (SSD). Phân loại dùng GAP trên C5.

In [ ]:
class EELANBackbone(nn.Module):
    """Stem(H/2) -> stage1(H/4) -> stage2=C3(H/8) -> stage3=C4(H/16) -> stage4=C5(H/32)."""
    def __init__(self, C_in=3, C_stem=32, widths=(64,128,192,256), m=2.0, g=2):
        super().__init__()
        w1, w2, w3, w4 = widths
        self.stem = nn.Sequential(
            ConvBNAct(C_in, C_stem, k=3, s=2),   # H/2
            ConvBNAct(C_stem, C_stem, k=3, s=1),
        )
        self.stage1 = nn.Sequential(
            ConvBNAct(C_stem, w1, k=3, s=2),     # H/4
            EELANLite(w1, w1, m=m, g=g),
        )
        self.stage2 = nn.Sequential(
            ConvBNAct(w1, w2, k=3, s=2),         # H/8   -> C3
            EELANLite(w2, w2, m=m, g=g),
        )
        self.stage3 = nn.Sequential(
            ConvBNAct(w2, w3, k=3, s=2),         # H/16  -> C4
            EELANLite(w3, w3, m=m, g=g),
        )
        self.stage4 = nn.Sequential(
            ConvBNAct(w3, w4, k=3, s=2),         # H/32  -> C5
            EELANLite(w4, w4, m=m, g=g),
        )
        self.out_channels = {'C3': w2, 'C4': w3, 'C5': w4}
        self._init_weights()

    def _init_weights(self):
        for mod in self.modules():
            if isinstance(mod, nn.Conv2d):
                nn.init.kaiming_normal_(mod.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(mod, nn.BatchNorm2d):
                nn.init.constant_(mod.weight, 1.0); nn.init.constant_(mod.bias, 0.0)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        c3 = self.stage2(x)
        c4 = self.stage3(c3)
        c5 = self.stage4(c4)
        return {'C3': c3, 'C4': c4, 'C5': c5}

class EELANClassifier(nn.Module):
    def __init__(self, num_classes, C_in=3, C_stem=32, widths=(64,128,192,256), m=2.0, g=2):
        super().__init__()
        self.backbone = EELANBackbone(C_in, C_stem, widths, m, g)
        c5 = self.backbone.out_channels['C5']
        self.head = nn.Linear(c5, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        z = F.adaptive_avg_pool2d(feats['C5'], 1).flatten(1)
        return self.head(z)

# Kiểm tra kích thước đặc trưng
_tmp = EELANBackbone(widths=CFG['widths'], m=CFG['m'], g=CFG['g'])
with torch.no_grad():
    _o = _tmp(torch.randn(1, 3, CFG['img_size'], CFG['img_size']))
for k, v in _o.items():
    print(k, tuple(v.shape))
del _tmp, _o

## 4. Dữ liệu (ImageFolder)

In [ ]:
def build_transforms(img_size=224, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=InterpolationMode.BILINEAR),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
    ])

def build_dataloaders(cfg):
    train_root = os.path.join(cfg['data_root'], cfg['train_dir'])
    val_root   = os.path.join(cfg['data_root'], cfg['val_dir'])
    assert os.path.isdir(train_root), f'Không thấy thư mục train: {train_root}'

    if os.path.isdir(val_root) and any(os.scandir(val_root)):
        train_ds = datasets.ImageFolder(train_root, build_transforms(cfg['img_size'], True))
        val_ds   = datasets.ImageFolder(val_root,   build_transforms(cfg['img_size'], False))
    else:
        base = datasets.ImageFolder(train_root)
        n_total = len(base)
        n_val = max(1, int(round(n_total * cfg['auto_split_val_ratio'])))
        n_train = n_total - n_val
        gen = torch.Generator().manual_seed(cfg['seed'])
        tr_idx, va_idx = torch.utils.data.random_split(range(n_total), [n_train, n_val], generator=gen)
        train_full = datasets.ImageFolder(train_root, build_transforms(cfg['img_size'], True))
        val_full   = datasets.ImageFolder(train_root, build_transforms(cfg['img_size'], False))
        train_ds = Subset(train_full, list(tr_idx))
        val_ds   = Subset(val_full,   list(va_idx))

    class_names = train_ds.dataset.classes if hasattr(train_ds, 'dataset') else train_ds.classes
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              num_workers=cfg['num_workers'], pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=cfg['num_workers'], pin_memory=True)
    return train_loader, val_loader, class_names

train_loader, val_loader, class_names = build_dataloaders(CFG)
num_classes = len(class_names)
print('Classes:', class_names, '| #train batch:', len(train_loader), '| #val batch:', len(val_loader))

## 5. Huấn luyện + lưu lịch sử (phục vụ Task 2)

In [ ]:
@torch.no_grad()
def accuracy_top1(logits, y):
    return (logits.argmax(1) == y).float().mean().item() * 100.0

def train_one_epoch(model, loader, optimizer, scaler, loss_fn):
    model.train()
    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                logits = model(x); loss = loss_fn(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            logits = model(x); loss = loss_fn(logits, y)
            loss.backward(); optimizer.step()
        bs = x.size(0)
        tot_loss += loss.item() * bs; tot_acc += accuracy_top1(logits, y) * bs; n += bs
    return tot_loss / n, tot_acc / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    tot_loss, tot_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x); loss = loss_fn(logits, y)
        bs = x.size(0)
        tot_loss += loss.item() * bs; tot_acc += accuracy_top1(logits, y) * bs; n += bs
    return tot_loss / n, tot_acc / n

In [ ]:
def train_model(cfg):
    model = EELANClassifier(num_classes, C_stem=cfg['C_stem'], widths=cfg['widths'],
                            m=cfg['m'], g=cfg['g']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    if cfg['use_cosine']:
        def lr_lambda(ep):
            if cfg['epochs'] <= 1: return 1.0
            t = ep / (cfg['epochs'] - 1)
            cos = 0.5 * (1 + math.cos(math.pi * t))
            return cfg['min_lr_scale'] + (1 - cfg['min_lr_scale']) * cos
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    else:
        scheduler = None
    loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    scaler = torch.cuda.amp.GradScaler() if (cfg['amp'] and device.type == 'cuda') else None

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_wts = -1.0, copy.deepcopy(model.state_dict())

    for ep in range(cfg['epochs']):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler, loss_fn)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        if scheduler is not None: scheduler.step()

        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)

        if va_acc > best_acc:
            best_acc = va_acc
            best_wts = copy.deepcopy(model.state_dict())
            torch.save({'model': best_wts, 'classes': class_names, 'cfg': cfg}, cfg['best_ckpt'])
            # Lưu riêng backbone để tái dùng cho detection
            torch.save({'backbone': model.backbone.state_dict(),
                        'classes': class_names,
                        'widths': cfg['widths'], 'C_stem': cfg['C_stem'],
                        'm': cfg['m'], 'g': cfg['g']}, cfg['backbone_ckpt'])

        print(f"Epoch {ep+1:02d}/{cfg['epochs']:02d} | "
              f"train {tr_loss:.4f}/{tr_acc:.2f}% | val {va_loss:.4f}/{va_acc:.2f}% | {time.time()-t0:.1f}s")

    model.load_state_dict(best_wts)
    print(f"\nBest val acc: {best_acc:.2f}%  -> {cfg['best_ckpt']}")
    print(f"Backbone saved -> {cfg['backbone_ckpt']}")
    return model, history

model, history = train_model(CFG)

## 6. Task 2a — Đồ thị biến thiên Loss & Accuracy

In [ ]:
epochs_axis = range(1, len(history['train_loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(epochs_axis, history['train_loss'], 'o-', label='train')
ax[0].plot(epochs_axis, history['val_loss'],   's-', label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(epochs_axis, history['train_acc'], 'o-', label='train')
ax[1].plot(epochs_axis, history['val_acc'],   's-', label='val')
ax[1].set_title('Accuracy (%)'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('acc (%)'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()
curve_path = os.path.join(RESULTS, 'cls_training_curves.png')
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu:', curve_path)

# Lưu history ra CSV
import csv
with open(os.path.join(RESULTS, 'cls_history.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['epoch','train_loss','train_acc','val_loss','val_acc'])
    for i in epochs_axis:
        w.writerow([i, history['train_loss'][i-1], history['train_acc'][i-1],
                    history['val_loss'][i-1], history['val_acc'][i-1]])
print('Đã lưu history CSV.')

## 7. Task 2b — Precision + Recall + Confusion Matrix (trên tập val có nhãn)

In [ ]:
from sklearn.metrics import (classification_report, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay)

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_y, all_p = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        all_p.append(logits.argmax(1).cpu().numpy())
        all_y.append(y.numpy())
    return np.concatenate(all_y), np.concatenate(all_p)

y_true, y_pred = collect_predictions(model, val_loader)

print('=== Classification report (per-class precision / recall / f1) ===')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
print(f'MACRO  precision={p:.4f}  recall={r:.4f}  f1={f1:.4f}')

# Lưu report ra file text
with open(os.path.join(RESULTS, 'cls_precision_recall.txt'), 'w', encoding='utf-8') as f:
    f.write(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    f.write(f'\nMACRO precision={p:.4f} recall={r:.4f} f1={f1:.4f}\n')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix (val)')
cm_path = os.path.join(RESULTS, 'cls_confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight'); plt.show()
print('Đã lưu:', cm_path)

## 8. Task 1 — Lưu / Nạp weights & Inference (1 ảnh hoặc cả thư mục)
`load_weights(path)` nạp từ đường dẫn chỉ định, dựng lại model đúng cấu hình đã lưu, rồi chạy dự đoán, in **class + xác suất**, và xuất CSV.

In [ ]:
def load_weights(ckpt_path):
    """Dựng lại EELANClassifier từ checkpoint và nạp weights."""
    ckpt = torch.load(ckpt_path, map_location=device)
    cfg_s = ckpt['cfg']; classes = ckpt['classes']
    net = EELANClassifier(len(classes), C_stem=cfg_s['C_stem'], widths=cfg_s['widths'],
                          m=cfg_s['m'], g=cfg_s['g']).to(device)
    net.load_state_dict(ckpt['model']); net.eval()
    print(f'Đã nạp weights từ {ckpt_path} | classes={classes}')
    return net, classes, cfg_s.get('img_size', 224)

@torch.no_grad()
def predict_path(net, classes, path, img_size=224, topk=3, save_csv=None):
    """path: 1 ảnh hoặc 1 thư mục ảnh. In class + xác suất; trả về list kết quả."""
    exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.webp','*.JPG','*.PNG','*.JPEG')
    if os.path.isdir(path):
        files = sorted(sum([glob.glob(os.path.join(path, e)) for e in exts], []))
    else:
        files = [path]
    tfm = build_transforms(img_size, is_train=False)
    rows = []
    for f in files:
        try:
            im = Image.open(f).convert('RGB')
        except Exception as e:
            print('[bỏ qua]', f, e); continue
        x = tfm(im).unsqueeze(0).to(device)
        prob = torch.softmax(net(x), dim=1)[0]
        k = min(topk, len(classes))
        tv, ti = prob.topk(k)
        preds = [(classes[i], float(tv[j])) for j, i in enumerate(ti)]
        top_cls, top_p = preds[0]
        print(f'\n{os.path.basename(f)}  ->  {top_cls} ({top_p*100:.1f}%)')
        for c, pp in preds:
            print(f'    {c:>8s}: {pp:.4f}')
        rows.append({'file': f, 'pred': top_cls, 'confidence': top_p,
                     'all_probs': {classes[i]: float(prob[i]) for i in range(len(classes))}})
    if save_csv:
        import csv
        with open(save_csv, 'w', newline='', encoding='utf-8') as fo:
            w = csv.writer(fo); w.writerow(['file', 'pred', 'confidence'] + [f'p_{c}' for c in classes])
            for row in rows:
                w.writerow([row['file'], row['pred'], f"{row['confidence']:.4f}"] +
                           [f"{row['all_probs'][c]:.4f}" for c in classes])
        print('\nĐã lưu CSV:', save_csv)
    return rows

In [ ]:
# --- Demo Task 1: nạp weights đã lưu và chạy inference ---
net, classes, img_size = load_weights(CFG['best_ckpt'])

# Đổi INFER_PATH thành 1 ảnh hoặc 1 thư mục ảnh bất kỳ.
# Mặc định chạy thử trên thư mục val/<class đầu tiên> nếu có.
INFER_PATH = os.path.join(DATA_CLS, CFG['val_dir'], classes[0])
if not os.path.isdir(INFER_PATH):
    INFER_PATH = os.path.join(DATA_CLS, CFG['train_dir'], classes[0])

rows = predict_path(net, classes, INFER_PATH, img_size=img_size, topk=CFG['topk'],
                    save_csv=os.path.join(RESULTS, 'cls_predictions.csv'))
print('\nTổng số ảnh dự đoán:', len(rows))

---
### Hoàn tất Notebook 01
Đầu ra:
- `weights/best_cls.pt`, `weights/backbone.pt`
- `results/cls_training_curves.png`, `results/cls_history.csv`
- `results/cls_precision_recall.txt`, `results/cls_confusion_matrix.png`
- `results/cls_predictions.csv`

→ Chuyển sang **Notebook 02** để xây mô hình phát hiện đối tượng (dùng `backbone.pt`).